# Attention Is All You Need — Implementation / 구현

**Paper**: Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. (2017). *Attention Is All You Need*. NeurIPS 30.

This notebook builds the Transformer from scratch — Scaled Dot-Product Attention, Multi-Head Attention, Positional Encoding, Layer Normalization, Position-wise FFN, full Encoder/Decoder layers — first in NumPy, then visualized and demonstrated with a toy task, and finally compared to PyTorch's built-in `nn.Transformer`.

이 노트북은 Transformer를 밑바닥부터 구현한다 — Scaled Dot-Product Attention, Multi-Head Attention, Positional Encoding, LayerNorm, Position-wise FFN, 전체 Encoder/Decoder layer — 먼저 NumPy로 만들고, 시각화하고, 장난감 task에서 시연한 뒤, PyTorch의 내장 `nn.Transformer`와 비교한다.

**Paper hyperparameters (base model) / 논문 하이퍼파라미터 (base)**:
- $N = 6$ layers, $d_{\text{model}} = 512$, $d_{ff} = 2048$, $h = 8$ heads, $d_k = d_v = 64$, dropout $= 0.1$

In this notebook we use smaller dimensions for clarity and fast execution. / 본 노트북에서는 명확성과 빠른 실행을 위해 더 작은 차원을 사용한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## Part 1: Scaled Dot-Product Attention (NumPy) / 스케일 내적 어텐션

The core operation of the Transformer.

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V
$$

- $Q \in \mathbb{R}^{n \times d_k}$ : $n$개의 query 벡터 / queries
- $K \in \mathbb{R}^{m \times d_k}$ : $m$개의 key 벡터 / keys
- $V \in \mathbb{R}^{m \times d_v}$ : $m$개의 value 벡터 / values
- Scaling by $\sqrt{d_k}$ keeps the dot-product variance at $O(1)$ so softmax does not saturate.
- $\sqrt{d_k}$ 스케일링은 내적의 분산을 $O(1)$로 유지하여 softmax가 포화되지 않게 한다.

In [ ]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax along a given axis.

    Args:
        x: Input array of arbitrary shape.
        axis: Axis along which softmax is applied.

    Returns:
        Array of the same shape with softmax applied along ``axis``.
    """
    x_shifted = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x_shifted)
    return e / np.sum(e, axis=axis, keepdims=True)


def scaled_dot_product_attention(
    Q: np.ndarray,
    K: np.ndarray,
    V: np.ndarray,
    mask: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Scaled dot-product attention.

    Computes ``softmax(Q K^T / sqrt(d_k)) V`` with optional additive mask.

    Args:
        Q: Query matrix of shape ``(..., n, d_k)``.
        K: Key matrix of shape ``(..., m, d_k)``.
        V: Value matrix of shape ``(..., m, d_v)``.
        mask: Optional boolean mask of shape ``(..., n, m)``. ``True`` entries
            are kept; ``False`` entries are set to ``-inf`` before softmax.

    Returns:
        A tuple ``(output, attention_weights)`` where ``output`` has shape
        ``(..., n, d_v)`` and ``attention_weights`` has shape ``(..., n, m)``.
    """
    d_k = Q.shape[-1]
    scores = np.matmul(Q, np.swapaxes(K, -1, -2)) / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores, axis=-1)
    output = np.matmul(weights, V)
    return output, weights


# Quick sanity check / 간단한 확인
n, m, d_k, d_v = 4, 6, 8, 16
Q = np.random.randn(n, d_k)
K = np.random.randn(m, d_k)
V = np.random.randn(m, d_v)
out, attn = scaled_dot_product_attention(Q, K, V)
print(f"output shape: {out.shape} (expected ({n}, {d_v}))")
print(f"attention shape: {attn.shape} (expected ({n}, {m}))")
print(f"row sums (should be 1): {attn.sum(axis=-1)}")

### Why divide by $\sqrt{d_k}$? — Variance experiment / 분산 실험

Without scaling, the variance of $q \cdot k$ grows linearly with $d_k$, pushing softmax into saturation.

스케일링이 없으면 $q \cdot k$의 분산이 $d_k$에 선형으로 커져 softmax가 포화 영역으로 밀려난다.

In [ ]:
d_k_values = [8, 32, 128, 512, 2048]
variances_raw = []
variances_scaled = []

for d_k in d_k_values:
    q = np.random.randn(10_000, d_k)
    k = np.random.randn(10_000, d_k)
    dot = (q * k).sum(axis=1)
    variances_raw.append(dot.var())
    variances_scaled.append((dot / np.sqrt(d_k)).var())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(d_k_values, variances_raw, 'o-', label='raw $q \\cdot k$')
ax[0].plot(d_k_values, d_k_values, '--', color='grey', label='y = $d_k$')
ax[0].set_xscale('log'); ax[0].set_yscale('log')
ax[0].set_xlabel('$d_k$'); ax[0].set_ylabel('variance')
ax[0].set_title('Without scaling / 스케일링 없음')
ax[0].legend()

ax[1].plot(d_k_values, variances_scaled, 'o-', color='C2')
ax[1].axhline(1.0, linestyle='--', color='grey', label='y = 1')
ax[1].set_xscale('log'); ax[1].set_ylim(0, 2)
ax[1].set_xlabel('$d_k$'); ax[1].set_ylabel('variance')
ax[1].set_title('Scaled by $1/\\sqrt{d_k}$ / 스케일링 적용')
ax[1].legend()
plt.tight_layout(); plt.show()

## Part 2: Multi-Head Attention (NumPy) / 멀티헤드 어텐션

$$
\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)
$$
$$
\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)\, W^O
$$

We project the inputs into $h$ sub-spaces of dimension $d_k = d_v = d_{\text{model}} / h$, run attention in parallel, then concatenate.

입력을 $d_k = d_v = d_{\text{model}} / h$ 차원의 $h$개 부분공간으로 투영하고 병렬로 attention을 수행한 뒤 연결한다.

In [ ]:
class MultiHeadAttention:
    """NumPy implementation of multi-head attention.

    Each head operates in a ``d_model / h`` dimensional subspace. The total
    computational cost is similar to a single-head attention with full model
    dimensionality, but representation is distributed across heads that can
    specialize in different relational patterns.

    Attributes:
        d_model: Total model (residual stream) dimensionality.
        h: Number of attention heads.
        d_k: Per-head key/query dimension.
        d_v: Per-head value dimension.
    """

    def __init__(self, d_model: int, h: int, seed: int = 0) -> None:
        assert d_model % h == 0, "d_model must be divisible by h"
        self.d_model = d_model
        self.h = h
        self.d_k = d_model // h
        self.d_v = d_model // h

        rng = np.random.default_rng(seed)
        scale = 1.0 / np.sqrt(d_model)
        self.W_q = rng.standard_normal((d_model, d_model)) * scale
        self.W_k = rng.standard_normal((d_model, d_model)) * scale
        self.W_v = rng.standard_normal((d_model, d_model)) * scale
        self.W_o = rng.standard_normal((d_model, d_model)) * scale

    def _split_heads(self, x: np.ndarray) -> np.ndarray:
        """Reshape ``(n, d_model)`` to ``(h, n, d_k)``."""
        n = x.shape[0]
        return x.reshape(n, self.h, self.d_k).transpose(1, 0, 2)

    def _merge_heads(self, x: np.ndarray) -> np.ndarray:
        """Reshape ``(h, n, d_v)`` back to ``(n, d_model)``."""
        h, n, d_v = x.shape
        return x.transpose(1, 0, 2).reshape(n, h * d_v)

    def __call__(
        self,
        Q: np.ndarray,
        K: np.ndarray,
        V: np.ndarray,
        mask: np.ndarray | None = None,
    ) -> tuple[np.ndarray, np.ndarray]:
        """Run multi-head attention.

        Args:
            Q: ``(n, d_model)`` query input.
            K: ``(m, d_model)`` key input.
            V: ``(m, d_model)`` value input.
            mask: Optional ``(n, m)`` boolean mask broadcastable across heads.

        Returns:
            Tuple ``(output, per_head_weights)`` with shapes
            ``(n, d_model)`` and ``(h, n, m)``.
        """
        Qp = self._split_heads(Q @ self.W_q)
        Kp = self._split_heads(K @ self.W_k)
        Vp = self._split_heads(V @ self.W_v)

        if mask is not None:
            mask = mask[None, :, :]  # broadcast to (h, n, m)
        heads_out, weights = scaled_dot_product_attention(Qp, Kp, Vp, mask)
        merged = self._merge_heads(heads_out)
        return merged @ self.W_o, weights


mha = MultiHeadAttention(d_model=64, h=8)
X = np.random.randn(10, 64)
out, w = mha(X, X, X)
print(f"output shape: {out.shape} (expected (10, 64))")
print(f"weights shape: {w.shape} (expected (8, 10, 10))")

## Part 3: Positional Encoding / 위치 인코딩

$$
PE_{(pos,\, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \qquad
PE_{(pos,\, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

Wavelengths form a geometric progression from $2\pi$ to $10000 \cdot 2\pi$. Each dimension pair acts like a clock hand of a different speed — the set of hands uniquely fingerprints every position.

파장은 $2\pi$에서 $10000 \cdot 2\pi$까지 기하급수적으로 증가. 각 차원 쌍은 서로 다른 속도의 시계 바늘처럼 동작하며, 그 집합이 각 위치의 고유한 지문을 만든다.

In [ ]:
def positional_encoding(max_len: int, d_model: int) -> np.ndarray:
    """Sinusoidal positional encoding as in Vaswani et al. (2017).

    Args:
        max_len: Maximum sequence length.
        d_model: Model (embedding) dimensionality.

    Returns:
        Array of shape ``(max_len, d_model)`` where even columns are sines and
        odd columns are cosines of geometrically-spaced frequencies.
    """
    pos = np.arange(max_len)[:, None]
    i = np.arange(d_model)[None, :]
    angle_rates = 1.0 / np.power(10000, (2 * (i // 2)) / d_model)
    angles = pos * angle_rates
    pe = np.zeros((max_len, d_model))
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return pe


pe = positional_encoding(max_len=100, d_model=64)
print(f"PE shape: {pe.shape}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im = axes[0].imshow(pe, aspect='auto', cmap='RdBu_r')
axes[0].set_xlabel('dimension $i$'); axes[0].set_ylabel('position $pos$')
axes[0].set_title('Positional Encoding heatmap / 히트맵')
plt.colorbar(im, ax=axes[0])

for dim in [0, 4, 16, 60]:
    axes[1].plot(pe[:, dim], label=f'dim {dim}')
axes[1].set_xlabel('position'); axes[1].set_ylabel('value')
axes[1].set_title('Period grows with dimension / 차원이 커질수록 주기 증가')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### Relative-position property / 상대 위치 성질

For any fixed offset $k$, $PE_{pos+k}$ is a linear function of $PE_{pos}$ (a rotation per dimension pair). This lets attention learn to attend by **relative** distance.

고정 offset $k$에 대해 $PE_{pos+k}$는 $PE_{pos}$의 선형 함수(각 차원 쌍의 회전). 덕분에 attention이 **상대** 거리 기반으로 학습할 수 있다.

We verify this by plotting the cosine similarity between $PE_{pos}$ and $PE_{pos+k}$ across positions — it depends only on $k$, not on $pos$ (for small $pos$ it is exact; for large $pos$ very nearly).

$PE_{pos}$와 $PE_{pos+k}$의 코사인 유사도가 $k$에만 의존하고 $pos$에는 거의 의존하지 않음을 확인한다.

In [ ]:
pe = positional_encoding(max_len=200, d_model=128)
norms = np.linalg.norm(pe, axis=1, keepdims=True)
pe_unit = pe / norms
sim = pe_unit @ pe_unit.T  # cosine similarity matrix

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
im = axes[0].imshow(sim, cmap='viridis')
axes[0].set_title('Cosine similarity of PE vectors / PE 코사인 유사도')
axes[0].set_xlabel('position'); axes[0].set_ylabel('position')
plt.colorbar(im, ax=axes[0])

# Same offset k from different base positions should give similar similarity
for base in [0, 20, 50, 100]:
    offsets = np.arange(0, 50)
    if base + offsets.max() < pe.shape[0]:
        sims = [sim[base, base + k] for k in offsets]
        axes[1].plot(offsets, sims, label=f'base = {base}')
axes[1].set_xlabel('offset $k$'); axes[1].set_ylabel('cosine similarity')
axes[1].set_title('Similarity depends on $k$, not on base / $k$에만 의존')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Part 4: Layer Normalization and Position-wise FFN / LayerNorm과 위치별 FFN

**LayerNorm** normalizes each token's features independently:
$$
\text{LayerNorm}(x) = \gamma \odot \frac{x - \mu}{\sigma + \varepsilon} + \beta
$$
where $\mu, \sigma$ are the mean and standard deviation over the feature axis.

**Position-wise FFN**:
$$
\text{FFN}(x) = \max(0,\, xW_1 + b_1)\, W_2 + b_2
$$

Applied identically at every position; equivalent to two 1×1 convolutions.

위치 간 가중치는 공유, layer 간에는 별개. 두 개의 1×1 convolution과 등가.

In [ ]:
class LayerNorm:
    """Layer normalization over the last axis.

    Attributes:
        gamma: Learned per-feature scale.
        beta: Learned per-feature shift.
        eps: Numerical stability constant.
    """

    def __init__(self, d_model: int, eps: float = 1e-6) -> None:
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
        self.eps = eps

    def __call__(self, x: np.ndarray) -> np.ndarray:
        mu = x.mean(axis=-1, keepdims=True)
        sigma = x.std(axis=-1, keepdims=True)
        return self.gamma * (x - mu) / (sigma + self.eps) + self.beta


class PositionwiseFFN:
    """Two-layer MLP with ReLU, applied per position.

    Equivalent to two 1x1 convolutions. The inner dimension ``d_ff`` is
    typically ``4 * d_model`` in the paper.
    """

    def __init__(self, d_model: int, d_ff: int, seed: int = 0) -> None:
        rng = np.random.default_rng(seed)
        s1, s2 = 1.0 / np.sqrt(d_model), 1.0 / np.sqrt(d_ff)
        self.W1 = rng.standard_normal((d_model, d_ff)) * s1
        self.b1 = np.zeros(d_ff)
        self.W2 = rng.standard_normal((d_ff, d_model)) * s2
        self.b2 = np.zeros(d_model)

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return np.maximum(0.0, x @ self.W1 + self.b1) @ self.W2 + self.b2


ln = LayerNorm(64)
ffn = PositionwiseFFN(64, 256)
x = np.random.randn(5, 64) * 3 + 7  # shifted and scaled input
y = ln(x)
print(f"After LayerNorm — mean: {y.mean(axis=-1)}, std: {y.std(axis=-1)}")
print(f"FFN output shape: {ffn(x).shape}")

## Part 5: Encoder and Decoder Layers / 인코더와 디코더 레이어

**Encoder layer** (2 sub-layers):
$$
x' = \text{LayerNorm}(x + \text{MHA}(x,x,x))
$$
$$
x'' = \text{LayerNorm}(x' + \text{FFN}(x'))
$$

**Decoder layer** (3 sub-layers): masked self-attention → cross-attention over encoder output → FFN, each wrapped by Add & Norm.

Decoder layer는 masked self-attention → encoder 출력에 대한 cross-attention → FFN 순으로, 각각 residual + LayerNorm으로 감싼다.

In [ ]:
class EncoderLayer:
    """A single Transformer encoder layer.

    Consists of multi-head self-attention and a position-wise FFN, each
    wrapped with residual connection followed by layer normalization.
    """

    def __init__(self, d_model: int, h: int, d_ff: int, seed: int = 0) -> None:
        self.mha = MultiHeadAttention(d_model, h, seed=seed)
        self.ffn = PositionwiseFFN(d_model, d_ff, seed=seed + 1)
        self.ln1 = LayerNorm(d_model)
        self.ln2 = LayerNorm(d_model)

    def __call__(self, x: np.ndarray, mask: np.ndarray | None = None) -> np.ndarray:
        attn_out, _ = self.mha(x, x, x, mask)
        x = self.ln1(x + attn_out)
        x = self.ln2(x + self.ffn(x))
        return x


class DecoderLayer:
    """A single Transformer decoder layer.

    Consists of masked multi-head self-attention, multi-head cross-attention
    over encoder output, and a position-wise FFN. Each sub-layer is wrapped
    with residual connection followed by layer normalization.
    """

    def __init__(self, d_model: int, h: int, d_ff: int, seed: int = 0) -> None:
        self.self_mha = MultiHeadAttention(d_model, h, seed=seed)
        self.cross_mha = MultiHeadAttention(d_model, h, seed=seed + 1)
        self.ffn = PositionwiseFFN(d_model, d_ff, seed=seed + 2)
        self.ln1 = LayerNorm(d_model)
        self.ln2 = LayerNorm(d_model)
        self.ln3 = LayerNorm(d_model)

    def __call__(
        self,
        x: np.ndarray,
        enc_out: np.ndarray,
        causal_mask: np.ndarray,
        cross_mask: np.ndarray | None = None,
    ) -> np.ndarray:
        self_out, _ = self.self_mha(x, x, x, causal_mask)
        x = self.ln1(x + self_out)
        cross_out, _ = self.cross_mha(x, enc_out, enc_out, cross_mask)
        x = self.ln2(x + cross_out)
        x = self.ln3(x + self.ffn(x))
        return x


def causal_mask(size: int) -> np.ndarray:
    """Lower-triangular boolean mask preventing attending to future positions.

    Args:
        size: Sequence length.

    Returns:
        Boolean array of shape ``(size, size)`` that is ``True`` on and below
        the diagonal.
    """
    return np.tril(np.ones((size, size), dtype=bool))


d_model, h, d_ff = 64, 8, 256
n_src, n_tgt = 10, 7
src = np.random.randn(n_src, d_model)
tgt = np.random.randn(n_tgt, d_model)

enc = EncoderLayer(d_model, h, d_ff, seed=1)
dec = DecoderLayer(d_model, h, d_ff, seed=2)
enc_out = enc(src)
dec_out = dec(tgt, enc_out, causal_mask=causal_mask(n_tgt))
print(f"encoder output: {enc_out.shape}")
print(f"decoder output: {dec_out.shape}")

## Part 6: Visualizing the causal mask and attention / 인과 마스크와 어텐션 시각화

The causal mask is the lower-triangular pattern that keeps training parallel while enforcing the autoregressive property at each position.

인과 마스크는 하삼각 패턴으로, 학습 병렬화와 위치별 autoregressive 속성을 동시에 보장한다.

In [ ]:
n = 10
mask = causal_mask(n)

# Unmasked vs masked attention on a random input
X = np.random.randn(n, 64)
mha_demo = MultiHeadAttention(d_model=64, h=4, seed=42)
_, attn_unmasked = mha_demo(X, X, X)
_, attn_masked = mha_demo(X, X, X, mask=mask)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(mask, cmap='gray_r')
axes[0].set_title('Causal mask / 인과 마스크\n(True = attend)')
axes[1].imshow(attn_unmasked[0], cmap='viridis')
axes[1].set_title('Attention without mask (head 0)')
axes[2].imshow(attn_masked[0], cmap='viridis')
axes[2].set_title('Attention with causal mask (head 0)')
for ax in axes:
    ax.set_xlabel('key position'); ax.set_ylabel('query position')
plt.tight_layout(); plt.show()

print(f"Row sums with mask (should be 1.0): {attn_masked[0].sum(axis=-1)}")

## Part 7: Full NumPy Transformer — Figure 1 end-to-end / 전체 NumPy Transformer — Figure 1 재현

We now assemble all building blocks into a **complete encoder-decoder Transformer** in NumPy, following Figure 1 of the paper exactly. This includes:

앞서 만든 블록들을 모두 조립하여 논문 Figure 1을 정확히 따르는 **완전한 encoder-decoder Transformer**를 NumPy로 만든다. 포함:

1. **Token Embedding** — integer id → $d_{\text{model}}$ vector, scaled by $\sqrt{d_{\text{model}}}$.
2. **Positional Encoding addition** — sinusoidal PE added to embeddings.
3. **Encoder stack** — $N$ encoder layers.
4. **Decoder stack** — $N$ decoder layers with causal + cross attention.
5. **Output linear + softmax** — project to vocabulary logits.

All of Figure 1 is reproduced here in pure NumPy (untrained — weights are random). The goal is to validate **shape and dataflow**, not accuracy.

무작위 가중치를 사용한 미학습 모델이다. 목표는 **shape와 데이터 흐름** 검증, 정확도 아님.

In [ ]:
class Embedding:
    """Token embedding table with paper's sqrt(d_model) scaling.

    Attributes:
        table: Learnable lookup table of shape ``(vocab_size, d_model)``.
        d_model: Embedding dimensionality.
    """

    def __init__(self, vocab_size: int, d_model: int, seed: int = 0) -> None:
        rng = np.random.default_rng(seed)
        self.table = rng.standard_normal((vocab_size, d_model)) * (1.0 / np.sqrt(d_model))
        self.d_model = d_model

    def __call__(self, token_ids: np.ndarray) -> np.ndarray:
        """Look up embeddings and scale by sqrt(d_model).

        Args:
            token_ids: Integer array of token ids, shape ``(n,)``.

        Returns:
            Array of shape ``(n, d_model)``.
        """
        return self.table[token_ids] * np.sqrt(self.d_model)


class Encoder:
    """Stack of ``N`` identical encoder layers.

    Each layer contains multi-head self-attention and a position-wise FFN,
    each wrapped with residual + LayerNorm.
    """

    def __init__(self, N: int, d_model: int, h: int, d_ff: int, seed: int = 0) -> None:
        self.layers = [
            EncoderLayer(d_model, h, d_ff, seed=seed + 10 * i) for i in range(N)
        ]

    def __call__(self, x: np.ndarray, mask: np.ndarray | None = None) -> np.ndarray:
        for layer in self.layers:
            x = layer(x, mask)
        return x


class Decoder:
    """Stack of ``N`` identical decoder layers.

    Each layer performs masked self-attention, cross-attention over the
    encoder memory, and a position-wise FFN.
    """

    def __init__(self, N: int, d_model: int, h: int, d_ff: int, seed: int = 0) -> None:
        self.layers = [
            DecoderLayer(d_model, h, d_ff, seed=seed + 10 * i) for i in range(N)
        ]

    def __call__(
        self,
        x: np.ndarray,
        enc_out: np.ndarray,
        causal_mask: np.ndarray,
        cross_mask: np.ndarray | None = None,
    ) -> np.ndarray:
        for layer in self.layers:
            x = layer(x, enc_out, causal_mask, cross_mask)
        return x


class TransformerNumPy:
    """End-to-end encoder-decoder Transformer in pure NumPy.

    Reproduces Figure 1 of Vaswani et al. (2017): embedding + positional
    encoding -> encoder stack; embedding + positional encoding -> decoder
    stack with cross-attention -> linear -> softmax.

    Input/output embeddings and the output linear projection share the same
    weight matrix (weight tying, as in the paper).
    """

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 64,
        N: int = 2,
        h: int = 4,
        d_ff: int = 256,
        max_len: int = 64,
        seed: int = 0,
    ) -> None:
        self.d_model = d_model
        self.embed = Embedding(vocab_size, d_model, seed=seed)
        self.pe = positional_encoding(max_len, d_model)
        self.encoder = Encoder(N, d_model, h, d_ff, seed=seed + 100)
        self.decoder = Decoder(N, d_model, h, d_ff, seed=seed + 200)
        # Weight tying: output projection shares embedding matrix (transposed)
        self.W_out = self.embed.table.T  # (d_model, vocab_size)

    def encode(self, src_ids: np.ndarray) -> np.ndarray:
        """Run encoder: ids -> memory of shape ``(n_src, d_model)``."""
        x = self.embed(src_ids) + self.pe[: len(src_ids)]
        return self.encoder(x)

    def decode(self, tgt_ids: np.ndarray, enc_out: np.ndarray) -> np.ndarray:
        """Run decoder: target ids + encoder memory -> decoder features.

        Returns shape ``(n_tgt, d_model)``.
        """
        x = self.embed(tgt_ids) + self.pe[: len(tgt_ids)]
        mask = causal_mask(len(tgt_ids))
        return self.decoder(x, enc_out, causal_mask=mask)

    def __call__(self, src_ids: np.ndarray, tgt_ids: np.ndarray) -> np.ndarray:
        """Full forward pass returning token probabilities.

        Args:
            src_ids: Source token ids, shape ``(n_src,)``.
            tgt_ids: Target (shifted-right) token ids, shape ``(n_tgt,)``.

        Returns:
            Probability array of shape ``(n_tgt, vocab_size)`` — softmax over
            the vocabulary at each decoder position.
        """
        enc_out = self.encode(src_ids)
        dec_out = self.decode(tgt_ids, enc_out)
        logits = dec_out @ self.W_out
        return softmax(logits, axis=-1)

### End-to-end forward pass (shape check) / 전체 forward pass (shape 검증)

We instantiate a small Transformer ($N=2$, $d_{\text{model}}=64$, $h=4$) and run an untrained forward pass. The goal is to confirm every tensor flows correctly from source ids to per-position vocabulary distribution — **exactly the Figure 1 pipeline**.

작은 Transformer를 만들어 미학습 forward pass를 실행한다. 소스 id에서 위치별 vocab 분포까지 모든 텐서가 올바르게 흐름을 확인 — **Figure 1 파이프라인 그대로**.

In [ ]:
VOCAB_SIZE = 50

model_np = TransformerNumPy(
    vocab_size=VOCAB_SIZE, d_model=64, N=2, h=4, d_ff=256, max_len=32, seed=0
)

# Fake source and (shifted-right) target sequences
src_ids = np.array([5, 12, 7, 22, 3, 18, 9])
tgt_ids = np.array([0, 14, 8, 25, 11])  # 0 == BOS

# Intermediate shapes
print('=== Figure 1 dataflow (NumPy) ===')
print(f'src_ids:            {src_ids.shape}   → {src_ids}')
src_emb = model_np.embed(src_ids) + model_np.pe[: len(src_ids)]
print(f'after embed + PE:   {src_emb.shape}')
enc_out = model_np.encode(src_ids)
print(f'encoder output Z:   {enc_out.shape}')
print()
print(f'tgt_ids:            {tgt_ids.shape}   → {tgt_ids}')
dec_out = model_np.decode(tgt_ids, enc_out)
print(f'decoder features:   {dec_out.shape}')
probs = model_np(src_ids, tgt_ids)
print(f'probabilities:      {probs.shape}  (n_tgt={len(tgt_ids)}, vocab={VOCAB_SIZE})')
print(f'row sums (must ≈1): {probs.sum(axis=-1)}')
print(f'argmax token ids:   {probs.argmax(axis=-1)}   ← untrained random weights')

### Greedy decoding with the NumPy model / NumPy 모델로 그리디 디코딩

We also run the **autoregressive inference loop** end-to-end in NumPy: encode once, then generate one token at a time using the causal mask. Outputs are meaningless (weights are random), but the machinery is the full Figure 1 inference path.

NumPy로 **autoregressive 추론 루프**도 끝까지 실행한다. 가중치가 무작위이므로 출력 자체는 무의미하나, Figure 1 추론 경로 전체가 동작함을 보여준다.

In [ ]:
def greedy_decode_numpy(
    model: TransformerNumPy,
    src_ids: np.ndarray,
    bos: int,
    eos: int,
    max_len: int = 16,
) -> list[int]:
    """Autoregressive greedy decoding with the NumPy Transformer.

    Encoder is run once; the decoder is re-run at every step with the
    accumulated target prefix and its causal mask.

    Args:
        model: A :class:`TransformerNumPy` instance.
        src_ids: Source token ids, shape ``(n_src,)``.
        bos: Beginning-of-sequence token id.
        eos: End-of-sequence token id (stops generation when produced).
        max_len: Hard cap on generated tokens (excluding BOS).

    Returns:
        List of generated token ids (excluding the leading BOS).
    """
    enc_out = model.encode(src_ids)
    tgt = [bos]
    for _ in range(max_len):
        dec_out = model.decode(np.array(tgt), enc_out)
        logits = dec_out[-1] @ model.W_out
        next_id = int(np.argmax(logits))
        tgt.append(next_id)
        if next_id == eos:
            break
    return tgt[1:]


generated = greedy_decode_numpy(
    model_np, src_ids, bos=0, eos=VOCAB_SIZE - 1, max_len=10
)
print(f'src:       {src_ids.tolist()}')
print(f'generated: {generated}   (random weights → meaningless content, pipeline works)')

## Part 8: PyTorch reference — tiny copy task / PyTorch 참조 — 작은 copy task

We now compare our NumPy understanding against PyTorch's built-in `nn.Transformer`. We train a tiny Transformer to **copy** an input sequence of integers (trivial but shows the full training loop works end-to-end).

PyTorch 내장 `nn.Transformer`와 비교하기 위해 정수 시퀀스를 **복사**하는 아주 작은 Transformer를 학습시킨다 (단순하지만 전체 학습 파이프라인이 동작함을 보여준다).

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)


class TinyTransformer(nn.Module):
    """A minimal encoder-decoder Transformer for sequence-to-sequence tasks.

    Args:
        vocab_size: Size of the shared source/target vocabulary.
        d_model: Model dimensionality.
        nhead: Number of attention heads.
        num_layers: Number of encoder and decoder layers.
        dim_ff: Inner dimension of the position-wise FFN.
        max_len: Maximum sequence length for positional encoding.
    """

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 2,
        dim_ff: int = 128,
        max_len: int = 32,
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        self.register_buffer(
            'pe',
            torch.tensor(positional_encoding(max_len, d_model), dtype=torch.float32),
        )
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_ff,
            dropout=0.0,
            batch_first=True,
        )
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, src: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        src_emb = self.embed(src) * np.sqrt(self.d_model) + self.pe[: src.size(1)]
        tgt_emb = self.embed(tgt) * np.sqrt(self.d_model) + self.pe[: tgt.size(1)]
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1))
        h = self.transformer(src_emb, tgt_emb, tgt_mask=tgt_mask)
        return self.out(h)


# Build a small copy dataset
VOCAB = 20  # tokens 0..19, reserve 0 for BOS and 19 for EOS
BOS, EOS = 0, 19
SEQ_LEN = 8


def make_batch(batch: int = 64) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Generate a batch of copy-task pairs.

    For each example we draw a random integer sequence of length ``SEQ_LEN``
    in ``[1, VOCAB - 2]``. The target is the same sequence prefixed with BOS
    and terminated with EOS.

    Returns:
        Tuple ``(src, tgt_input, tgt_output)`` each of shape
        ``(batch, SEQ_LEN[+1])``.
    """
    content = torch.randint(1, VOCAB - 1, (batch, SEQ_LEN))
    src = content
    tgt_in = torch.cat([torch.full((batch, 1), BOS), content], dim=1)
    tgt_out = torch.cat([content, torch.full((batch, 1), EOS)], dim=1)
    return src, tgt_in, tgt_out


model = TinyTransformer(VOCAB)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

losses = []
for step in range(400):
    src, tgt_in, tgt_out = make_batch(64)
    logits = model(src, tgt_in)
    loss = loss_fn(logits.reshape(-1, VOCAB), tgt_out.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if step % 50 == 0:
        print(f"step {step:4d}  loss {loss.item():.4f}")

plt.plot(losses); plt.xlabel('step'); plt.ylabel('cross-entropy')
plt.title('Copy-task training loss / 복사 task 학습 손실')
plt.grid(True, alpha=0.3); plt.show()

### Greedy autoregressive decoding / 그리디 자기회귀 디코딩

Now we show the **inference-time** behaviour — starting from BOS, we generate one token at a time until EOS.

이제 **추론 시점** 동작을 보여준다 — BOS에서 시작해 EOS까지 한 토큰씩 생성한다.

In [ ]:
@torch.no_grad()
def greedy_decode(model: TinyTransformer, src: torch.Tensor, max_len: int = 10) -> list[int]:
    """Autoregressively decode one sequence using greedy argmax.

    Args:
        model: Trained tiny Transformer.
        src: Source tensor of shape ``(1, L)``.
        max_len: Maximum number of tokens to generate (excluding BOS).

    Returns:
        List of generated token ids, stopping at EOS if produced.
    """
    model.eval()
    tgt = torch.tensor([[BOS]])
    for _ in range(max_len):
        logits = model(src, tgt)
        next_id = int(logits[0, -1].argmax().item())
        tgt = torch.cat([tgt, torch.tensor([[next_id]])], dim=1)
        if next_id == EOS:
            break
    return tgt[0, 1:].tolist()


# Test on a few fresh sequences
print("src -> predicted (should equal src then EOS)")
for _ in range(5):
    src = torch.randint(1, VOCAB - 1, (1, SEQ_LEN))
    pred = greedy_decode(model, src, max_len=SEQ_LEN + 2)
    match = pred[:SEQ_LEN] == src[0].tolist()
    print(f"  src: {src[0].tolist()}  pred: {pred}  exact-copy: {match}")

### Learning-rate schedule from the paper / 논문의 학습률 스케줄

$$
lr = d_{\text{model}}^{-0.5} \cdot \min(\text{step}^{-0.5},\; \text{step} \cdot \text{warmup}^{-1.5})
$$

Linearly increasing during warmup, then inverse-square-root decay.
Warmup 동안 선형 증가, 이후 역제곱근으로 감소.

In [ ]:
def noam_schedule(step: int, d_model: int = 512, warmup: int = 4000) -> float:
    """Noam learning-rate schedule from Vaswani et al. (2017)."""
    step = max(step, 1)
    return d_model ** -0.5 * min(step ** -0.5, step * warmup ** -1.5)


steps = np.arange(1, 20_000)
lrs = [noam_schedule(s) for s in steps]
plt.plot(steps, lrs)
plt.axvline(4000, color='red', linestyle='--', alpha=0.5, label='warmup = 4000')
plt.xlabel('step'); plt.ylabel('learning rate')
plt.title('Noam LR schedule ($d_{model}=512$, warmup=4000)')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 (Vaswani 2017) | Modern Equivalent / 현대 등가물 |
|---|---|---|
| Attention score | Scaled dot-product, $\sqrt{d_k}$ scaling | FlashAttention (memory-efficient, same math) |
| Multi-head | $h=8$ heads, $d_k=64$ | Grouped-Query / Multi-Query Attention (LLaMA-2, Mistral) |
| Positional encoding | Sinusoidal, additive | RoPE (rotary), ALiBi (linear bias) |
| Normalization | Post-LN (after residual add) | Pre-LN and RMSNorm (GPT-2+, LLaMA) |
| Activation in FFN | ReLU | GELU (BERT), SwiGLU (PaLM/LLaMA) |
| Decoding | Beam search + length penalty | Top-$k$ / top-$p$ + temperature sampling |
| Training | 6-layer, ~65M params, WMT | Scaling laws (Kaplan 2020), 100B+ params |

### Connections to other papers in this reading list / 관련 논문

- **#17 Bahdanau 2014** — additive cross-attention; Transformer generalizes to dot-product.
- **#21 Ba et al. 2016** — Layer Normalization, directly used here.
- **#20 He et al. 2015** — Residual connection pattern, essential for deep stacks.
- **#28 Devlin 2018 (BERT)** — encoder-only pre-training built on this architecture.
- **#31 Dosovitskiy 2020 (ViT)** — applies the same block to image patches.

### Take-home / 요점

The Transformer is built from a small number of composable primitives — attention, FFN, residual, LayerNorm, positional encoding. Everything in the modern foundation-model stack (GPT, BERT, LLaMA, ViT, CLIP, ...) rearranges or scales these pieces without fundamentally replacing them.

Transformer는 몇 가지 조립 가능한 기본 블록으로 구성된다 — attention, FFN, residual, LayerNorm, positional encoding. 현대 foundation model 스택(GPT, BERT, LLaMA, ViT, CLIP 등)은 이 블록들을 재배열하거나 규모를 키울 뿐 근본적으로 대체하지 않는다.